In [30]:
from collections import defaultdict
from collections import Counter

In [31]:

class SemanticCache:
    """
    A simple semantic cache implementation using a dictionary to store query-answer pairs.
    """
    def __init__(self, threshold=0.8):
        self.cache = defaultdict(str)
        self.threshold = threshold

    def _cosine_similarity(self, vec1, vec2):
        """Calculate the cosine similarity between two vectors."""
        dot_product = sum(a * b for a, b in zip(vec1, vec2))
        magnitude_vec1 = sum(a ** 2 for a in vec1) ** 0.5
        magnitude_vec2 = sum(b ** 2 for b in vec2) ** 0.5
        if magnitude_vec1 == 0 or magnitude_vec2 == 0:
            return 0.0
        return dot_product / (magnitude_vec1 * magnitude_vec2)

    def add_query_answer_to_cache(self, query, answer):
        self.cache[query] = answer
        return self.cache

    def _get_vocab(self, query, cache_store):
        cached_queries = list(cache_store.keys())
        vocab  = list(set(query.lower().split() + [w for q in cached_queries for w in q.lower().split()]))
        return vocab
    
    def _vectorize(self, text, vocab):
        """
        Example vectorization function that converts text into a bag-of-words vector based on the provided vocabulary.
        """
        word_count = Counter(text.lower().split())
        return [word_count.get(word, 0) for word in vocab]

    def get_cache(self):
        return self.cache

    def get_answer_from_cache(self, query):
        cache_store = self.get_cache()
        vocab = self._get_vocab(query, cache_store)
        query_vector = self._vectorize(query, vocab)
        list_query_vector = [query_vector] * len(cache_store)
        list_vect_cache_queries = [self._vectorize(cached_query, vocab) for cached_query in cache_store.keys()]

        scores = list(map(lambda q, q_cache: self._cosine_similarity(q, q_cache), 
                          [q for q in list_query_vector],
                          [q_cache for q_cache in list_vect_cache_queries]))
        
        best_score_index = scores.index(max(scores))
        if scores[best_score_index] >= self.threshold:
            best_matching_query = list(cache_store.keys())[best_score_index]
            return cache_store[best_matching_query]

    def del_response_from_cache(self, cached_response):
        if cached_response in self.cache.values():
            query_to_delete = [key for key, value in self.cache.items() if value == cached_response][0]
            del self.cache[query_to_delete]

In [32]:
semantic_cache = SemanticCache(threshold=0.8)

semantic_cache.add_query_answer_to_cache(query='What is the capital of France?', 
                                         answer='The capital of France is Paris.')

semantic_cache.add_query_answer_to_cache(query='What is the largest mammal?',
                                         answer='The largest mammal is the blue whale.')

defaultdict(str,
            {'What is the capital of France?': 'The capital of France is Paris.',
             'What is the largest mammal?': 'The largest mammal is the blue whale.'})

In [33]:
cached_answer = semantic_cache.get_answer_from_cache('What is the capital of France?')

if cached_answer:
    print(f"Cached answer: {cached_answer}")
else:
    print("No suitable cached answer found.")

Cached answer: The capital of France is Paris.


In [34]:
cached_answer = semantic_cache.get_answer_from_cache('Where are the Eiffel Tower and the Louvre located?')

if cached_answer:
    print(f"Cached answer: {cached_answer}")
else:
    print("No suitable cached answer found.")

No suitable cached answer found.


In [35]:
# add a new query-answer pair to the cache
_ = semantic_cache.add_query_answer_to_cache(query='Where are the Eiffel Tower and the Louvre located?',
                                         answer='The Eiffel Tower and the Louvre are located in Paris, France.')


In [36]:
semantic_cache.get_cache()

defaultdict(str,
            {'What is the capital of France?': 'The capital of France is Paris.',
             'What is the largest mammal?': 'The largest mammal is the blue whale.',
             'Where are the Eiffel Tower and the Louvre located?': 'The Eiffel Tower and the Louvre are located in Paris, France.'})